# 📈 Machine Learning for Financial Analysis
**Authors:** Prakhar Singhal [0131CL221071], Priyanshu Saxena [0131CL221078]  
**Institution:** Jai Narain College of Technology, Bhopal  
**Session:** 2022–2026  

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from copy import copy
from scipy import stats
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error

print('Libraries imported successfully ✅')

## 2. Load Dataset

In [ ]:
stocks_df = pd.read_csv('data/stock.csv')
stocks_df = stocks_df.sort_values(by=['Date']).reset_index(drop=True)
stocks_df

## 3. Dataset Info & Stock Names

In [ ]:
stocks_df.info()
print(f'\nTotal Number of stocks: {len(stocks_df.columns[1:])}')
print('Stocks under consideration:')
for i in stocks_df.columns[1:]:
    print(' ', i)

## 4. Check Null Values & Descriptive Stats

In [ ]:
print('Null values:\n', stocks_df.isnull().sum())
print('\nDescriptive Statistics:')
stocks_df.describe()

## 5. Raw Stock Price Plot

In [ ]:
def show_plot(df, fig_title):
    df.plot(x='Date', figsize=(15, 7), linewidth=2, title=fig_title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

show_plot(stocks_df, 'RAW STOCK PRICES (WITHOUT NORMALIZATION)')

## 6. Normalized Stock Prices

In [ ]:
def normalize(df):
    x = df.copy()
    for i in x.columns[1:]:
        x[i] = x[i] / x[i].iloc[0]
    return x

show_plot(normalize(stocks_df), 'NORMALIZED STOCK PRICES')

## 7. Interactive Visualization (Plotly)

In [ ]:
def interactive_plot(df, title):
    fig = px.line(title=title)
    for i in df.columns[1:]:
        fig.add_scatter(x=df['Date'], y=df[i], name=i)
    fig.show()

interactive_plot(stocks_df, 'Stock Prices')

## 8. Daily Returns

In [ ]:
def daily_return(df):
    df_daily_return = df.copy()
    for i in df.columns[1:]:
        for j in range(1, len(df)):
            df_daily_return[i].iloc[j] = ((df[i].iloc[j] - df[i].iloc[j-1]) / df[i].iloc[j-1]) * 100
        df_daily_return[i].iloc[0] = 0
    return df_daily_return

stocks_daily_return = daily_return(stocks_df)
stocks_daily_return

## 9. Correlation Heatmap

In [ ]:
cm = stocks_daily_return.drop(columns=['Date']).corr()
plt.figure(figsize=(10, 8))
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt='.2f', cmap='RdPu', ax=ax)
plt.title('Correlation Heatmap of Daily Returns')
plt.tight_layout()
plt.show()

## 10. Histograms of Daily Returns

In [ ]:
stocks_daily_return.drop(columns=['Date']).hist(figsize=(12, 10), bins=40)
plt.suptitle('Daily Return Distributions', y=1.01)
plt.tight_layout()
plt.show()

## 11. Model Training & Evaluation

In [ ]:
target_variable = 'AAPL'

def create_features(df, target):
    features = df[['Date', target]].copy()
    features['prev_close'] = features[target].shift(1)
    features.dropna(inplace=True)
    return features.drop('Date', axis=1)

data = create_features(stocks_df.copy(), target_variable)
X = data.drop(target_variable, axis=1)
y = data[target_variable]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Support Vector Machine': SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
}

results = {}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    results[model_name] = score
    print(f'{model_name:30s}  R²: {score:.4f}   MAE: {mae:.4f}')

## 12. Model R² Score Comparison

In [ ]:
plt.figure(figsize=(9, 5))
bars = plt.bar(results.keys(), results.values(),
               color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
plt.ylim(0.99, 1.0005)
plt.title('Model R² Score Comparison')
plt.ylabel('R² Score')
plt.xticks(rotation=15, ha='right')
for bar, score in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.00005,
             f'{score:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()